In [ ]:
import csv
import sys
import pandas as pd
sys.path.append('/home/unix/vkrhk/EmmaEmb/EmmaEmb/analysis')
from constants import IMG_OUTPUT_PATH, EMB_SPACES

PRECOMPUTED_KNN_PATH = f'{IMG_OUTPUT_PATH}/knn-binding-sites'
def read_kNN(k, emb_space_name, metric, do_average=True):
    results = {}
    with open(f'{PRECOMPUTED_KNN_PATH}/{metric}/k={k}.csv', 'r') as f:
        knn_results = csv.reader(f, delimiter=',')
        header = next(knn_results)
        emb_space_index = header.index(emb_space_name)
        for row in knn_results:
            feature, acc = row[0], float(row[emb_space_index])
            class_name, number_of_points = feature.split(' ')[0], int(feature.split(' ')[-1][:-1])
            results[class_name] = (acc, number_of_points)
        
    if do_average:
        results = sum(float(a) * n for a, n in results.values()) / sum(n for _, n in results.values())
        return results
    else:
        return results

print(read_kNN(3, 'ESM2', 'euclidean'))
print(read_kNN(5, 'ESM2', 'euclidean'))
print(read_kNN(10, 'ESM2', 'euclidean'))
print(read_kNN(50, 'ESM2', 'euclidean'))
print(read_kNN(100, 'ESM2', 'euclidean'))
print(read_kNN(200, 'ESM2', 'euclidean'))

0.589807530807068
0.5697208082677773
0.5407758753663428
0.4754085042932799
0.455654789449329
0.441333359041596


In [51]:
import plotly.express as px
def plot_kNN_alignment_scores(fig):
    fig.update_layout(
            yaxis_title="Mean KNN feature<br>alignment score",
            template="plotly_white",
            font={"family": "Arial", "color": "black", "size": 12},
            legend_title_text="Embedding models",
            width=1200,
            height=500,
    )

    fig.update_traces(marker=dict(size=10), line=dict(width=4))

    for axis in fig.layout:
        if axis.startswith('xaxis'):
            fig.layout[axis].update(
                        showgrid=False,
                        linecolor='black',
                        linewidth=3,
                        ticks='outside',
                        tickwidth=2,
                        tickcolor='black',
                        ticklen=6,
                        tickformat=".0f"
                    )
        if axis.startswith('yaxis'):
            fig.layout[axis].update(
                showgrid=False,
                linecolor='black',
                linewidth=3,
                ticks='outside',
                tickwidth=2,
                tickcolor='black',
                ticklen=6,
                tickformat=".2f",
                dtick=0.01
            )    

    for annotation in fig.layout.annotations:
        if '=' in annotation.text:
            annotation.text = annotation.text.split('=')[1]
                
    fig.update_layout(
        title=dict(
            x=0,
            xanchor='left'
        ),
        legend=dict(
            orientation='h',
            x=0.48,
            y=1.05,
            xanchor='center',
            yanchor='bottom'
        ),
        margin=dict(t=120)
    )
    return fig


In [ ]:
import pickle
import plotly.colors as pc
import plotly.express as px

from constants import EMBEDDINGS_PATH, EMB_SPACES

Ks = [3, 5, 10, 50, 100, 200]

# read precomputed kNN scores
collected_results = []
for metric in ['euclidean', 'cosine', 'cityblock']:
    for k in Ks:
        for emb_space in EMB_SPACES:
            collected_results.append((k, emb_space[0], read_kNN(k, emb_space[0], metric), metric))

df = pd.DataFrame(collected_results, columns=['k', 'Embedding Space', 'kNN score', 'metric'])

# read precomputed MLP performance
performance_results = {}
for emb_space in EMB_SPACES:
    with open(f'{IMG_OUTPUT_PATH}/performance/train2-epochs=10/{emb_space[0]}_torch.pkl', 'rb') as f:
        performance_results[emb_space[0]] = pickle.load(f)

# combine kNN scores with MLP performance for legend text and color coding
new_collected_results = []
for result in collected_results:
    emb_space = result[1]
    performance = float(performance_results[emb_space]['test_auprc'])
    legend_text = f'{performance:.3f} ({emb_space})'
    new_collected_results.append(result + (legend_text, performance))
df = pd.DataFrame(new_collected_results, columns=['k', 'Embedding Space', 'kNN score', 'metric', 'Legend text', 'Performance'])

# get line color acccording to the MLP performance 
sorted_performance = df[['Performance', 'Legend text', 'Embedding Space']].drop_duplicates().sort_values('Performance', ascending=True)
sorted_performance_descending = df[['Performance', 'Legend text', 'Embedding Space']].drop_duplicates().sort_values('Performance', ascending=False)
n_bins = len(sorted_performance)
low, high = 0.3, 1.0
color_scale = pc.sample_colorscale(
        "Reds",
        [low + (high - low) * i / (n_bins - 1) for i in range(n_bins)],
        colortype='rgb'
        )
label_to_color ={}
for label, color in zip(sorted_performance['Legend text'].tolist(), color_scale):
    label_to_color[label] = color

# plot
fig = px.line(
    df,
    x="k",
    y="kNN score",
    color="Legend text",
    color_discrete_map=label_to_color,
    facet_col="metric",
    markers=True,
    category_orders={"Legend text": sorted_performance_descending['Legend text'].tolist()}
)
fig = plot_kNN_alignment_scores(fig)
fig.show()